# Cross-Validation & Model Selection

## Setup

In [1]:
# Setup

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.dummy import DummyRegressor

## Load Data

In [2]:
# load the clean datasets

df = pd.read_csv("data_train.csv")
df.sample()
df.columns

Index(['census_tract', 'StCoFIPS2019', 'StAbbr', 'walkability_index',
       'Pop2018', 'HU2018', 'HH2018', 'employment_mix',
       'employment_residential_mix', 'intersection_density',
       'transit_accessibility', 'employment_mix_ranked',
       'employment_residential_mix_ranked', 'intersection_density_ranked',
       'transit_accessibility_ranked', 'median_income', 'percent_unemployed',
       'percent_below_poverty', 'percent_bachelor_and_higher',
       'percent_over_65', 'percent_commute_car', 'percent_commute_transit',
       'percent_white', 'percent_black', 'percent_native_american',
       'percent_asian', 'percent_pacific_islander', 'statedesc', 'countyname',
       'total_population', 'arthritis_crudeprev', 'arthritis_crude95ci',
       'high_blood_pressure_prevalence', 'high_blood_pressure_95ci',
       'cancer_prevalence', 'cancer_95ci', 'current_asthma_prevalence',
       'current_asthma_95ci', 'coronary_heart_disease_prevalence',
       'coronary_heart_disease_95ci'

## Model cross-validation

### Configuration

In [3]:
features = [
    "employment_mix_ranked",
    "employment_residential_mix_ranked",
    "intersection_density_ranked",
    "transit_accessibility_ranked",
]

outcomes = [
    "high_cholesterol_prevalence",
    "depression_prevalence",
    "high_blood_pressure_prevalence",
    "obesity_prevalence",
    "coronary_heart_disease_prevalence",
    "cancer_prevalence",
]

group_col = "StCoFIPS2019"
n_splits = 5

### Pre-processing steps

In [4]:
# Plain: impute missing values with median strategy and apply standard scaler
pre_plain = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
    ]), features)
], remainder="drop")


# Interactions: impute + pairwise interactions + standard scaler
#   4 mains feature + 6 interaction terms = 10 features total
pre_inter = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
        ("sc", StandardScaler()),
    ]), features)
], remainder="drop")

### Pipelines

In [5]:
# Base models
pipe_linear = Pipeline([("pre", pre_plain), ("lr", LinearRegression())])
pipe_pca_linear = Pipeline([("pre", pre_plain), ("pca", PCA()), ("lr", LinearRegression())])
pipe_ridge  = Pipeline([("pre", pre_plain), ("ridge", Ridge())])
pipe_lasso  = Pipeline([("pre", pre_plain), ("lasso", Lasso(max_iter=10000))])

# Interaction models
pipe_linear_inter = Pipeline([("pre", pre_inter), ("lr", LinearRegression())])
pipe_ridge_inter = Pipeline([("pre", pre_inter), ("ridge", Ridge())])
pipe_lasso_inter = Pipeline([("pre", pre_inter), ("lasso", Lasso(max_iter=10000))])
pipe_pca_inter = Pipeline([("pre", pre_inter), ("pca", PCA()), ("lr", LinearRegression())])

### Hyper-parameters

In [6]:
grid_pca_linear = {"pca__n_components": [2, 3, 4]}
grid_ridge = {"ridge__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
grid_lasso = {"lasso__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
grid_pca_inter = {"pca__n_components": [2, 3, 4, 5, 6, 7, 8, 9, 10]} # After interactions we have 10 features

### Cross-validation

In [8]:
# GroupKFold splits our data set so that different folds have disjoint county codes
gkf = GroupKFold(n_splits=n_splits)

# Helper: convert neg MSE scores to RMSE mean & sd
"""
cross_validate has a scoring = "neg_mean_squared_error" option. Since we are intereste in the root mean squared error,
we create a small function that takes the score (the list of neg_mean_squared_error across splits) and return the mean
and standard deviation of the root mean squared errors.
"""
def mean_rmse_from_neg_mse(scores):
    mses = -np.array(scores, dtype=float)
    rmses = np.sqrt(mses)
    return float(rmses.mean()), float(rmses.std())

# Results rows
"""
We create an empty row, which will be a list of dictionaries that we can turn into a data set at the end of our 
cross validation.
""" 
rows = []

"""
We apply our cross validation strategy to each outcome.
"""
for target in outcomes:
    cols_needed = features + [target, group_col]
    d = df.dropna(subset=cols_needed).copy() # we drop the rows with missing values in cols_needed
    if d.empty: # if there are no rows with missing values, we append our rows data set with "No rows after dropna" and continue
        rows.append({"target": target, "model": "N/A", "rmse_mean": np.nan, "rmse_sd": np.nan, "details": "No rows after dropna"})
        continue

    X = d[features]
    y = d[target]
    groups = d[group_col]

    # 0) Dummy baseline (predicts mean per train fold)
    cv_dummy = cross_validate(
        DummyRegressor(strategy="mean"),
        X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring="neg_mean_squared_error",
        n_jobs=-1, return_train_score=False
    )
    rmse_mean, rmse_sd = mean_rmse_from_neg_mse(cv_dummy["test_score"])
    rows.append({"target": target, "model": "dummy", "rmse_mean": rmse_mean, "rmse_sd": rmse_sd, "details": ""})

    # 1a) Linear
    cv_lin = cross_validate(
        pipe_linear, X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring="neg_mean_squared_error",
        n_jobs=-1, return_train_score=False
    )
    rmse_mean, rmse_sd = mean_rmse_from_neg_mse(cv_lin["test_score"])
    rows.append({"target": target, "model": "linear", "rmse_mean": rmse_mean, "rmse_sd": rmse_sd, "details": ""})

    # 1b) Linear + interactions
    cv_lin_inter = cross_validate(
        pipe_linear_inter, X, y,
        cv=gkf.split(X, y, groups=groups),
        scoring="neg_mean_squared_error",
        n_jobs=-1, return_train_score=False
    )
    rmse_mean, rmse_sd = mean_rmse_from_neg_mse(cv_lin_inter["test_score"])
    rows.append({"target": target, "model": "linear + inter", "rmse_mean": rmse_mean, "rmse_sd": rmse_sd, "details": ""})

    # 2a) PCA + Linear (tune n_components)
    gs_pca_linear = GridSearchCV(
        pipe_pca_linear, grid_pca_linear,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_pca_linear.fit(X, y)
    rows.append({
        "target": target, "model": "PCA + linear",
        "rmse_mean": float(np.sqrt(-gs_pca_linear.best_score_)), "rmse_sd": np.nan,
        "details": f"best n_components={gs_pca_linear.best_params_.get('pca__n_components')}"
    })

    # 2b) PCA + Linear + inter (tune n_components)
    gs_pca_inter = GridSearchCV(
        pipe_pca_inter, grid_pca_inter,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_pca_inter.fit(X, y)
    rows.append({
        "target": target, "model": "PCA + linear + inter",
        "rmse_mean": float(np.sqrt(-gs_pca_inter.best_score_)), "rmse_sd": np.nan,
        "details": f"best n_components={gs_pca_inter.best_params_.get('pca__n_components')}"
    })

    # 3a) Ridge (tune alpha)
    gs_ridge = GridSearchCV(
        pipe_ridge, grid_ridge,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_ridge.fit(X, y)
    rows.append({
        "target": target, "model": "Ridge",
        "rmse_mean": float(np.sqrt(-gs_ridge.best_score_)), "rmse_sd": np.nan,
        "details": f"best alpha={gs_ridge.best_params_.get('ridge__alpha')}"
    })

    # 3b) Ridge + inter (tune alpha)
    gs_ridge_inter = GridSearchCV(
        pipe_ridge_inter, grid_ridge,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_ridge_inter.fit(X, y)
    rows.append({
        "target": target, "model": "Ridge + inter",
        "rmse_mean": float(np.sqrt(-gs_ridge_inter.best_score_)), "rmse_sd": np.nan,
        "details": f"best alpha={gs_ridge_inter.best_params_.get('ridge__alpha')}"
    })

    # 4a) Lasso (tune alpha)
    gs_lasso = GridSearchCV(
        pipe_lasso, grid_lasso,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_lasso.fit(X, y)
    rows.append({
        "target": target, "model": "Lasso",
        "rmse_mean": float(np.sqrt(-gs_lasso.best_score_)), "rmse_sd": np.nan,
        "details": f"best alpha={gs_lasso.best_params_.get('lasso__alpha')}"
    })

    # 4b) Lasso + inter (tune alpha)
    gs_lasso_inter = GridSearchCV(
        pipe_lasso_inter, grid_lasso,
        scoring="neg_mean_squared_error",
        cv=gkf.split(X, y, groups=groups),
        n_jobs=-1
    )
    gs_lasso_inter.fit(X, y)
    rows.append({
        "target": target, "model": "Lasso + inter",
        "rmse_mean": float(np.sqrt(-gs_lasso_inter.best_score_)), "rmse_sd": np.nan,
        "details": f"best alpha={gs_lasso_inter.best_params_.get('lasso__alpha')}"
    })

# Summarize & save
results = pd.DataFrame(rows).sort_values(["target", "rmse_mean"])
print(results)
results.to_csv("results_cv_by_outcome.csv", index=False)
print("\nSaved results to results_cv_by_outcome.csv")


                               target                 model  rmse_mean  \
47                  cancer_prevalence        linear + inter   1.721509   
51                  cancer_prevalence         Ridge + inter   1.721827   
49                  cancer_prevalence  PCA + linear + inter   1.721875   
53                  cancer_prevalence         Lasso + inter   1.721882   
46                  cancer_prevalence                linear   1.721985   
50                  cancer_prevalence                 Ridge   1.722342   
52                  cancer_prevalence                 Lasso   1.722347   
48                  cancer_prevalence          PCA + linear   1.722347   
45                  cancer_prevalence                 dummy   1.867657   
38  coronary_heart_disease_prevalence        linear + inter   1.925665   
42  coronary_heart_disease_prevalence         Ridge + inter   1.926046   
40  coronary_heart_disease_prevalence  PCA + linear + inter   1.926050   
44  coronary_heart_disease_prevalence 